In [1]:
import pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as sf
from pyspark.sql.window import Window

In [2]:
spark = SparkSession.Builder().getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/11 08:52:58 WARN Utils: Your hostname, Bayards-Macbook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.132.207 instead (on interface en0)
26/02/11 08:52:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/11 08:52:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.createDataFrame(pd.DataFrame({"A": [0,1,None], "B": ["Z", "", "null"]}))

In [4]:
shuffled_partitions = Window.partitionBy(*df.columns).orderBy(df.columns[0])
rank_column = "ranker"
initial_sdf = df.withColumn(rank_column, sf.row_number().over(shuffled_partitions))

In [7]:
sdf = initial_sdf.withColumn(
            "ID", sf.hex(sf.to_json(sf.array(*initial_sdf.columns)).cast("string"))
        ).drop(rank_column)

In [8]:
sdf.show()

26/02/11 08:53:19 ERROR Executor: Exception in task 0.0 in stage 2.0 (TID 10)10]
org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value 'Z' of the type "STRING" cannot be cast to "DOUBLE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"array" was called from
java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)

	at org.apache.spark.sql.errors.QueryExecutionErrors$.invalidInputInCastToNumberError(QueryExecutionErrors.scala:147)
	at org.apache.spark.sql.errors.QueryExecutionErrors.invalidInputInCastToNumberError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage3.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegen

NumberFormatException: [CAST_INVALID_INPUT] The value 'Z' of the type "STRING" cannot be cast to "DOUBLE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"array" was called from
java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)


In [9]:
new_sdf=initial_sdf
for column in initial_sdf.columns:
    new_sdf = new_sdf.withColumn(column, sf.col(column).cast("string"))

In [12]:
new_sdf.printSchema()

root
 |-- A: string (nullable = true)
 |-- B: string (nullable = true)
 |-- ranker: string (nullable = false)



In [13]:
sdf = new_sdf.withColumn(
            "ID", sf.hex(sf.to_json(sf.array(*new_sdf.columns)).cast("string"))
        ).drop(rank_column)

In [14]:
sdf.collect()

+---+----+--------------------+
|  A|   B|                  ID|
+---+----+--------------------+
|0.0|   Z|5B22302E30222C225...|
|1.0|    |5B22312E30222C222...|
|NaN|null|5B224E614E222C226...|
+---+----+--------------------+



In [10]:
sdf.withColumn("ID", sf.array(*initial_sdf.columns).cast("string")).show()

{"ts": "2026-02-11 08:53:30.525", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `ranker` cannot be resolved. Did you mean one of the following? [`A`, `B`, `ID`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o82.withColumn.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `ranker` cannot be resolved. Did you mean one of the following? [`A`, `B`, `ID`]. SQLSTATE: 42703;\n'Project [A#0, B#1, cast('array(A#0, B#1, 'ranker) as string) AS ID#18]\n+- Project [A#0, B#1, ID#4]\n   +- Project [A#0, B#1, ranker#2, hex(cast(to_json(array(A#0, cast(B#1 as double), cast(ranker#2

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `ranker` cannot be resolved. Did you mean one of the following? [`A`, `B`, `ID`]. SQLSTATE: 42703;
'Project [A#0, B#1, cast('array(A#0, B#1, 'ranker) as string) AS ID#18]
+- Project [A#0, B#1, ID#4]
   +- Project [A#0, B#1, ranker#2, hex(cast(to_json(array(A#0, cast(B#1 as double), cast(ranker#2 as double)), Some(America/Los_Angeles)) as string)) AS ID#4]
      +- Project [A#0, B#1, ranker#2]
         +- Project [A#0, B#1, ranker#2, ranker#2]
            +- Window [row_number() windowspecdefinition(A#0, B#1, A#0 ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS ranker#2], [A#0, B#1], [A#0 ASC NULLS FIRST]
               +- Project [A#0, B#1]
                  +- LogicalRDD [A#0, B#1], false


In [8]:
import pyspark
pyspark.__version__

'4.1.1'

In [21]:
print(*sdf.columns)

A B ID
